# N25 — Pace Agent

The Pace Agent answers one question: **how fast are we going this lap?**

It wraps the N06 XGBoost delta-lap-time model into a clean agent interface. Given the current race state (tyre compound, tyre life, lap number, track conditions, team/driver), it returns a lap time prediction with delta signals and a bootstrap confidence interval.

This is the first of seven sub-agents in the multi-agent strategy system (N25–N31). Its output feeds directly into the Strategy Orchestrator (N31) and into N28 (Pit Strategy Agent) as the pace signal used to evaluate undercut/overcut windows and stint extension decisions.

## Pipeline position

<pre>
race_state (lap_number, tyre_life, compound, team, ...)
    │
    ▼
N25 — Pace Agent  (LangGraph ReAct)
    │
    ├── lap_time_pred        absolute predicted lap time (seconds)
    ├── delta_vs_prev        delta vs previous lap (raw model output)
    ├── delta_vs_median      delta vs session median for this GP/compound
    └── ci_p10 / ci_p90      bootstrap confidence interval
</pre>

## Models used
- **N06** — XGBoost delta lap time (`data/models/lap_time/`)

## Steps
- **Step 0** — Setup, imports & model loading
- **Step 1** — Feature preparation (encoding maps + `build_lap_state`)
- **Step 2** — Inference functions + bootstrap confidence interval
- **Step 3** — `run_pace_agent(lap_state) → PaceOutput`
- **Step 4** — Demo: full 2025 race replay (lap by lap)
- **Step 5** — LangGraph ReAct agent (`@tool` wrappers + `create_react_agent`)
- **Step 6** — Export schema & config

---

In [1]:
# ── Step 0: Setup & model loading ──────────────────────────────────────────
import sys
import time
from pathlib import Path
from dataclasses import dataclass

repo_root = Path.cwd()
while not (repo_root / '.git').exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import json
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import fastf1
from IPython.display import clear_output

from langchain_core.tools import tool as lc_tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

# ── Paths ───────────────────────────────────────────────────────────────────
MODELS_DIR    = repo_root / 'data' / 'models' / 'lap_time'
PROCESSED_DIR = repo_root / 'data' / 'processed'
OUTPUTS_DIR   = repo_root / 'notebooks' / 'agents' / 'outputs'
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Load N06 model + features ───────────────────────────────────────────────
def load_pace_model(models_dir=MODELS_DIR):
    """Load the N06 XGBoost lap time delta model and its feature name list.

    Separates model loading from inference so the expensive I/O happens once
    at notebook setup time. Returns both artifacts together to ensure the
    feature list is always consistent with the model version on disk — callers
    must not reorder or drop features between load and predict.

    Args:
        models_dir: Path to the directory containing xgb_laptime_delta_final.json
            and xgb_laptime_delta_feature_names.json. Defaults to the global
            MODELS_DIR resolved from the repo root.

    Returns:
        Tuple (model, features) where model is a fitted xgb.XGBRegressor and
        features is a list of column name strings in the exact order the model
        expects at predict time.
    """
    features = json.loads((models_dir / 'xgb_laptime_delta_feature_names.json').read_text())
    model = xgb.XGBRegressor()
    model.load_model(models_dir / 'xgb_laptime_delta_final.json')
    return model, features

In [3]:
MODEL, FEATURES = load_pace_model()

print(f"Model loaded — {len(FEATURES)} features")
print(f"Features: {FEATURES}")

Model loaded — 25 features
Features: ['DriverNumber', 'LapNumber', 'Stint', 'TyreLife', 'FreshTyre', 'Position', 'CompoundID', 'TeamID', 'LapsSincePitStop', 'FuelLoad', 'Year', 'FuelEffect', 'Prev_LapTime', 'Prev_TyreLife', 'Prev_SpeedST', 'AirTemp', 'TrackTemp', 'Humidity', 'Rainfall', 'laps_remaining', 'Cluster', 'mean_sector_speed', 'Prev_DegradationRate', 'Prev_CumulativeDeg', 'Prev_DegAcceleration']


### Step 0 output

Model loaded correctly from `data/models/lap_time/`. The 25 features match the training set exactly — compound, team and circuit cluster are encoded as integers; degradation features (`Prev_DegradationRate`, `Prev_CumulativeDeg`, `Prev_DegAcceleration`) are included and will default to 0.0 on the first lap of a stint.

---

## Step 1 — Feature preparation

The N06 model expects 25 features. This step defines the encoding maps (compound, team, circuit cluster) and a `build_lap_state()` function that packs raw race state values into a single-row DataFrame ready for inference.

A few details worth noting:

- **`FreshTyre`** — binary flag, true when `tyre_life ≤ 1` (first flying lap on a new set)
- **`FuelEffect`** — approximated as `fuel_load × 0.03s/kg`, same convention used in N06
- **`laps_remaining`** — computed from `total_laps − lap_number`; used by the model to capture end-of-race pace changes
- **`mean_sector_speed`** — when not available directly, `Prev_SpeedST` is used as a proxy
- **Degradation features** (`Prev_DegradationRate`, `Prev_CumulativeDeg`, `Prev_DegAcceleration`) — default to 0.0 on the first lap of a stint; the demo in Step 4 will populate these from the actual stint history


In [4]:
# ── Step 1: Feature preparation (race state → model input) ─────────────────

CLUSTER_PARQUET  = PROCESSED_DIR / 'circuit_clustering' / 'circuit_clusters_k4_2025.parquet'
LAPS_FEATURED    = PROCESSED_DIR / 'laps_featured_2025.parquet'
FEATURE_MANIFEST = PROCESSED_DIR / 'feature_manifest_laptime.json'

In [5]:
# ── Load encoding maps from saved artifacts ─────────────────────────────────
def load_encoding_maps():
    """Load the three categorical encoding maps used by build_lap_state.

    Reads the compound encoding from the N06 feature manifest, the circuit
    cluster assignments from the k=4 clustering parquet (N05), and the
    team-to-integer map derived from the laps_featured parquet. All three maps
    are static artifacts from training — they must not be recomputed at inference
    time to avoid encoding drift between train and serve.

    Returns:
        Tuple (compound_id, circuit_cluster, team_id) where:
        - compound_id (dict): maps Pirelli compound string → integer code.
        - circuit_cluster (dict): maps GP_Name string → cluster integer (0–3).
        - team_id (dict): maps team name string → integer code used by the N06 model.
    """
    manifest    = json.loads(FEATURE_MANIFEST.read_text())
    compound_id = manifest['categorical_encoding']['Compound']

    clusters_df = pd.read_parquet(CLUSTER_PARQUET)[['GP_Name', 'Cluster']]
    circuit_cluster = dict(zip(clusters_df['GP_Name'], clusters_df['Cluster'].astype(int)))

    laps = pd.read_parquet(LAPS_FEATURED, columns=['Team', 'TeamID']).dropna()
    team_id = (laps.drop_duplicates('Team')
                    .set_index('Team')['TeamID']
                    .astype(int)
                    .to_dict())

    return compound_id, circuit_cluster, team_id

In [6]:
COMPOUND_ID, CIRCUIT_CLUSTER, TEAM_ID = load_encoding_maps()

print("Compound encoding:", COMPOUND_ID)
print("\nTeam encoding:", TEAM_ID)
print("\nCircuit → Cluster:")
for gp, c in sorted(CIRCUIT_CLUSTER.items()):
    print(f"  {gp:<22} → {c}")

Compound encoding: {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2, 'INTERMEDIATE': 3, 'WET': 4}

Team encoding: {'Red Bull Racing': 1, 'Alpine': 6, 'Mercedes': 2, 'Aston Martin': 5, 'Ferrari': 3, 'Williams': 7, 'Kick Sauber': 0, 'Racing Bulls': 0, 'Haas F1 Team': 10, 'McLaren': 4}

Circuit → Cluster:
  Austin                 → 1
  Baku                   → 1
  Barcelona              → 1
  Budapest               → 1
  Imola                  → 1
  Jeddah                 → 1
  Las Vegas              → 0
  Lusail                 → 0
  Marina Bay             → 1
  Melbourne              → 0
  Mexico City            → 3
  Miami                  → 1
  Monaco                 → 3
  Montréal               → 2
  Monza                  → 1
  Sakhir                 → 0
  Shanghai               → 1
  Silverstone            → 0
  Spa-Francorchamps      → 0
  Spielberg              → 1
  Suzuka                 → 0
  São Paulo              → 0
  Yas Island             → 1
  Zandvoort              → 0


### Step 1 output — encoding maps

All three encoding maps loaded from saved artifacts (no hardcoding):

- **Compound** — 5 values (SOFT→0 … WET→4), from `feature_manifest_laptime.json`
- **Team** — 10 teams; note `Kick Sauber` and `Racing Bulls` both map to 0 (merged in N06 training due to rebranding)
- **Circuit → Cluster** — 24 circuits across 4 K-Means clusters; Barcelona=1 (medium-speed), Monaco/Mexico=3 (street/high-load), Silverstone/Spa=0 (high-speed)

In [7]:
# ── build_lap_state ─────────────────────────────────────────────────────────
def build_lap_state(
    driver_number, lap_number, stint, tyre_life, compound,
    position, team, laps_since_pit, fuel_load, year,
    prev_lap_time, prev_tyre_life, prev_speed_st,
    air_temp, track_temp, humidity, rainfall,
    total_laps, gp_name,
    mean_sector_speed=None,
    prev_deg_rate=0.0, prev_cum_deg=0.0, prev_deg_accel=0.0,
):
    """Pack raw race state into a single-row DataFrame ready for MODEL.predict().

    Encodes categorical inputs (compound, team, circuit cluster) using the global
    lookup maps loaded from the training parquet, derives computed features
    (FreshTyre, FuelEffect, laps_remaining), and selects columns in the exact
    order FEATURES expects. The output is a one-row DataFrame with no missing
    values — callers should validate inputs before calling this function.

    Compound is encoded via COMPOUND_ID; team via TEAM_ID; circuit via
    CIRCUIT_CLUSTER. Unknown values fall back to default encodings (compound→1,
    team→0, cluster→1) so inference degrades gracefully on unseen categories.

    mean_sector_speed defaults to prev_speed_st when None — this covers the
    case where circuit_features data is unavailable for the current GP.

    Args:
        driver_number: Car number (int) for TeamID lookup.
        lap_number: Current race lap number (1-indexed).
        stint: Stint number (1-indexed).
        tyre_life: Laps on the current tyre set.
        compound: Pirelli compound string ('SOFT', 'MEDIUM', 'HARD', etc.).
        position: Current race position (1-based integer).
        team: Team name string matching TEAM_ID keys.
        laps_since_pit: Integer laps elapsed since the last pit stop.
        fuel_load: Fuel fraction in [0, 1] estimated as laps_remaining/total_laps.
        year: Race year integer (2023/2024/2025).
        prev_lap_time: Previous lap time in seconds.
        prev_tyre_life: TyreLife on the previous lap.
        prev_speed_st: Speed trap (km/h) from the previous lap.
        air_temp: Air temperature in °C.
        track_temp: Track surface temperature in °C.
        humidity: Relative humidity in %.
        rainfall: Boolean; True if rain was recorded.
        total_laps: Total scheduled race laps.
        gp_name: GP name string matching CIRCUIT_CLUSTER keys.
        mean_sector_speed: Average sector speed in km/h; defaults to prev_speed_st.
        prev_deg_rate: Degradation rate (s/lap) at the previous lap.
        prev_cum_deg: Cumulative degradation (s) at the previous lap.
        prev_deg_accel: Degradation acceleration (s/lap²) at the previous lap.

    Returns:
        Single-row pd.DataFrame with columns in FEATURES order, ready for
        MODEL.predict(). All values are numeric; no NaN cells.
    """
    compound_id    = COMPOUND_ID.get(compound, 1)
    team_id        = TEAM_ID.get(team, 0)
    cluster        = CIRCUIT_CLUSTER.get(gp_name, 1)
    fresh_tyre     = int(tyre_life <= 1)
    fuel_effect    = fuel_load * 0.03
    laps_remaining = max(0, total_laps - lap_number)
    mss            = mean_sector_speed if mean_sector_speed is not None else prev_speed_st

    row = {
        'DriverNumber':          driver_number,
        'LapNumber':             lap_number,
        'Stint':                 stint,
        'TyreLife':              tyre_life,
        'FreshTyre':             fresh_tyre,
        'Position':              position,
        'CompoundID':            compound_id,
        'TeamID':                team_id,
        'LapsSincePitStop':      laps_since_pit,
        'FuelLoad':              fuel_load,
        'Year':                  year,
        'FuelEffect':            fuel_effect,
        'Prev_LapTime':          prev_lap_time,
        'Prev_TyreLife':         prev_tyre_life,
        'Prev_SpeedST':          prev_speed_st,
        'AirTemp':               air_temp,
        'TrackTemp':             track_temp,
        'Humidity':              humidity,
        'Rainfall':              rainfall,
        'laps_remaining':        laps_remaining,
        'Cluster':               cluster,
        'mean_sector_speed':     mss,
        'Prev_DegradationRate':  prev_deg_rate,
        'Prev_CumulativeDeg':    prev_cum_deg,
        'Prev_DegAcceleration':  prev_deg_accel,
    }
    return pd.DataFrame([row])[FEATURES]

In [8]:
# ── Sanity check ─────────────────────────────────────────────────────────────
sample = build_lap_state(
    driver_number=44, lap_number=10, stint=1, tyre_life=10,
    compound='MEDIUM', position=3, team='Mercedes', laps_since_pit=10,
    fuel_load=90, year=2025, prev_lap_time=92.4, prev_tyre_life=9,
    prev_speed_st=310.0, air_temp=28.0, track_temp=42.0,
    humidity=40.0, rainfall=0.0, total_laps=57, gp_name='Barcelona',
)
print(sample.to_string())


   DriverNumber  LapNumber  Stint  TyreLife  FreshTyre  Position  CompoundID  TeamID  LapsSincePitStop  FuelLoad  Year  FuelEffect  Prev_LapTime  Prev_TyreLife  Prev_SpeedST  AirTemp  TrackTemp  Humidity  Rainfall  laps_remaining  Cluster  mean_sector_speed  Prev_DegradationRate  Prev_CumulativeDeg  Prev_DegAcceleration
0            44         10      1        10          0         3           1       2                10        90  2025         2.7          92.4              9         310.0     28.0       42.0      40.0       0.0              47        1              310.0                   0.0                 0.0                   0.0


### Step 1 output — `build_lap_state` sanity check

The 25-feature row is correctly assembled: `CompoundID=1` (MEDIUM), `TeamID=2` (Mercedes), `Cluster=1` (Barcelona), `FuelEffect=2.7` (90kg × 0.03), `laps_remaining=47` (57−10), degradation features at 0.0 (lap 1 of stint). Ready for `MODEL.predict()`.

---

## Step 2 — Inference + confidence interval

With the feature vector ready, this step handles the actual prediction and adds an uncertainty estimate around it.

**Single-point prediction** adds the model's predicted delta to `Prev_LapTime` to get an absolute lap time. The N06 model outputs a delta (±seconds vs previous lap), not an absolute time.

**Confidence interval** uses a lightweight bootstrap: the input row is perturbed N=200 times with 2% relative Gaussian noise on the continuous features most subject to real-world variability (lap time, speed trap, temperatures, tyre life). The P10/P90 of the resulting predictions gives a plausible range reflecting sensor noise and lap-to-lap variation — not formal model uncertainty, but sufficient for strategy decisions.

**Delta vs median** contextualises the raw prediction against the historical median for the same GP/year/compound from `laps_featured_2025.parquet`.

In [ ]:
# ── Step 2: Inference + bootstrap confidence interval ──────────────────────

N_BOOTSTRAP = 200
NOISE_PCT    = 0.02   # Gaussian noise scale: 2 % of feature value, approximates sensor noise / lap-to-lap variation

def predict_lap_time(lap_state_df, model=MODEL):
    """Predict the absolute lap time for the current lap.

    The N06 XGBoost model predicts a signed delta vs the previous lap
    (Prev_LapTime), not an absolute time. This function adds the delta back
    to Prev_LapTime so callers always receive an absolute lap time in seconds —
    which is the unit expected by PaceOutput and N31 Monte Carlo simulation.

    Args:
        lap_state_df: Single-row pd.DataFrame produced by build_lap_state(),
            with columns in FEATURES order. Prev_LapTime must be present.
        model: Trained xgb.Booster (default: global MODEL). Accepts a DMatrix
            internally; xgboost predict() handles the DataFrame conversion.

    Returns:
        Absolute predicted lap time in seconds (float), equal to
        Prev_LapTime + model_delta.
    """
    delta = float(model.predict(lap_state_df)[0])
    prev  = float(lap_state_df['Prev_LapTime'].iloc[0])
    return prev + delta


def bootstrap_confidence_interval(lap_state_df, model=MODEL, n=N_BOOTSTRAP, seed=42):
    """Estimate a P10/P90 confidence interval around the lap time prediction.

    Runs n forward passes of the model, each time adding independent Gaussian
    noise (σ = NOISE_PCT × feature value) to the continuous features most subject
    to real-world variability: Prev_LapTime, Prev_SpeedST, mean_sector_speed,
    AirTemp, TrackTemp, and TyreLife. The noise scale approximates typical
    sensor noise and lap-to-lap variation; it is not formal Bayesian uncertainty.
    P10/P90 of the resulting predictions bound the plausible prediction range
    and are used by N31 to sample pace scenarios in Monte Carlo strategy evaluation.

    Args:
        lap_state_df: Single-row pd.DataFrame from build_lap_state(), same
            as passed to predict_lap_time().
        model: Trained xgb.Booster (default: global MODEL).
        n: Number of bootstrap samples (default N_BOOTSTRAP = 200). Higher
            values reduce variance in the percentile estimate at the cost of
            compute time.
        seed: Integer seed for the numpy random generator, ensuring
            reproducible intervals across calls.

    Returns:
        Tuple (p10, p90) where both values are absolute lap times in seconds
        (float). p10 is the 10th percentile of the bootstrapped predictions;
        p90 is the 90th percentile.
    """
    rng = np.random.default_rng(seed)
    noise_cols = ['Prev_LapTime', 'Prev_SpeedST', 'mean_sector_speed',
                  'AirTemp', 'TrackTemp', 'TyreLife']

    base    = lap_state_df.values.copy().astype(float)
    col_idx = {c: lap_state_df.columns.get_loc(c) for c in noise_cols}

    preds = []
    for _ in range(n):
        row = base.copy()
        for col, idx in col_idx.items():
            sigma = abs(base[0, idx]) * NOISE_PCT
            row[0, idx] += rng.normal(0, sigma)
        df_row = pd.DataFrame(row, columns=lap_state_df.columns)
        delta  = float(model.predict(df_row)[0])
        preds.append(float(df_row['Prev_LapTime'].iloc[0]) + delta)

    return float(np.percentile(preds, 10)), float(np.percentile(preds, 90))


def session_median_lap_time(gp_name, year, compound, laps_df):
    """Compute the representative median lap time for a GP / year / compound.

    Filters laps_df to the matching GP name, year, and compound, then returns
    the median of LapTime_s. The median is a robust baseline that is
    unaffected by SC/VSC outlier laps (those laps are excluded from LAPS_REF
    at load time). N31 uses this value to contextualise the absolute predicted
    lap time — a large positive delta_vs_median signals the driver is on a
    degrading tyre or a slower compound.

    Args:
        gp_name: GP name string matching the GP_Name column in laps_df
            (e.g. 'Barcelona', 'Sakhir').
        year: Race year integer (2023/2024/2025).
        compound: Pirelli compound string ('SOFT', 'MEDIUM', 'HARD', etc.).
        laps_df: Reference pd.DataFrame loaded from laps_featured_2025.parquet,
            containing at minimum GP_Name, Year, Compound, and LapTime_s columns.

    Returns:
        Median lap time in seconds (float), or None when no matching laps
        exist in laps_df for the given combination.
    """
    mask = (
        (laps_df['GP_Name']  == gp_name) &
        (laps_df['Year']     == year) &
        (laps_df['Compound'] == compound)
    )
    subset = laps_df.loc[mask, 'LapTime_s'].dropna()
    return float(subset.median()) if len(subset) > 0 else None


# ── Load reference laps for median computation ───────────────────────────────
LAPS_REF = pd.read_parquet(
    PROCESSED_DIR / 'laps_featured_2025.parquet',
    columns=['GP_Name', 'Year', 'Compound', 'LapTime_s']
)
print(f"LAPS_REF loaded: {len(LAPS_REF)} rows")

### Step 2 output — LAPS_REF

Reference parquet loaded for median computation. Contains `GP_Name`, `Year`, `Compound` and `LapTime_s` only — minimal footprint, loaded once and reused across all `session_median_lap_time` calls.

In [10]:
# ── Sanity check ─────────────────────────────────────────────────────────────
pred     = predict_lap_time(sample)
p10, p90 = bootstrap_confidence_interval(sample)
median   = session_median_lap_time('Barcelona', 2025, 'MEDIUM', LAPS_REF)

delta_pred = pred - float(sample['Prev_LapTime'].iloc[0])

print(f"Predicted lap time : {pred:.3f}s")
print(f"Predicted delta    : {delta_pred:+.3f}s  (vs prev lap)")
print(f"Confidence interval: [{p10:.3f}s, {p90:.3f}s]")
print(f"Session median     : {median:.3f}s")
print(f"Delta vs median    : {pred - median:+.3f}s")


Predicted lap time : 92.803s
Predicted delta    : +0.403s  (vs prev lap)
Confidence interval: [90.106s, 95.278s]
Session median     : 80.959s
Delta vs median    : +11.845s


---

## Step 3 — `run_pace_agent(lap_state) → PaceOutput`

This step defines the agent's public interface: a single function that takes a lap state dictionary and returns a structured `PaceOutput` object.

`PaceOutput` is a dataclass with five fields:

- **`lap_time_pred`** — absolute predicted lap time (seconds)
- **`delta_vs_prev`** — delta vs the previous lap (`Prev_LapTime`), the raw model output
- **`delta_vs_median`** — delta vs the historical median for this GP/compound; contextualises whether the pace is fast or slow relative to what's expected here
- **`ci_p10` / `ci_p90`** — bootstrap confidence interval bounds

`run_pace_agent()` wraps `build_lap_state()` + `predict_lap_time()` + `bootstrap_confidence_interval()` into one call. This is the entry point that N31 (Strategy Orchestrator) will call.


In [11]:
# ── Step 3: run_pace_agent(lap_state) → PaceOutput ─────────────────────────

@dataclass
class PaceOutput:
    """Structured output of the Pace Agent for one lap.

    lap_time_pred is the XGBoost N06 prediction in absolute seconds — the model
    outputs a delta vs Prev_LapTime internally; this field adds Prev_LapTime back
    so all downstream agents work in absolute lap time.

    delta_vs_prev is the raw model delta (predicted lap_time minus Prev_LapTime).
    Negative means the driver is faster than the previous lap.

    delta_vs_median is the difference between lap_time_pred and the historical
    session median for this GP/year/compound combination. NaN when no median
    reference is available (new circuits or sparse data).

    ci_p10 and ci_p90 are the P10/P90 bootstrap confidence bounds on
    lap_time_pred, computed over N_BOOTSTRAP perturbations of the continuous
    input features. N31 Monte Carlo simulation samples from this interval to
    model pace uncertainty across strategy candidates.

    reasoning is a human-readable summary forwarded verbatim to the N31
    Orchestrator for LLM synthesis. Format:
    "Lap N: predicted X.XXXs (+/-Xs, faster/slower than prev). CI [X.X–X.Xs]. X.XXXs vs median."
    """
    lap_time_pred:   float
    delta_vs_prev:   float
    delta_vs_median: float
    ci_p10:          float
    ci_p90:          float
    reasoning:       str = ""  # forwarded to N31 Orchestrator


def run_pace_agent(
    driver_number, lap_number, stint, tyre_life, compound,
    position, team, laps_since_pit, fuel_load, year,
    prev_lap_time, prev_tyre_life, prev_speed_st,
    air_temp, track_temp, humidity, rainfall,
    total_laps, gp_name,
    mean_sector_speed=None,
    prev_deg_rate=0.0, prev_cum_deg=0.0, prev_deg_accel=0.0,
) -> PaceOutput:
    """Run the Pace Agent for a single lap and return a structured PaceOutput.

    Builds the full N06 feature vector from raw race state, calls the XGBoost
    model, computes a bootstrap P10/P90 uncertainty interval, and looks up the
    historical session median for the current GP/year/compound. Returns a
    PaceOutput with all fields populated and a reasoning string for N31.

    Args:
        driver_number: Car number used to look up TeamID encoding.
        lap_number: Current race lap; used for FuelLoad = laps_remaining/total_laps.
        stint: Stint number (1-indexed), forwarded as a raw feature.
        tyre_life: Laps on current tyre set; drives FreshTyre flag and CompoundID encoding.
        compound: Pirelli compound name ('SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET').
        position: Current race position (1-based).
        team: Team name matching TEAM_ID encoding map (e.g. 'McLaren').
        laps_since_pit: Laps since most recent pit stop.
        fuel_load: Estimated fuel fraction in [0, 1] (laps_remaining / total_laps).
        year: Race year (2023/2024/2025) used as a categorical feature.
        prev_lap_time: Previous lap time in seconds; model predicts a delta on top of this.
        prev_tyre_life: TyreLife on the previous lap (Prev_TyreLife feature).
        prev_speed_st: Speed trap reading in km/h from the previous lap.
        air_temp: Air temperature in °C from FastF1 weather data.
        track_temp: Track surface temperature in °C from FastF1 weather data.
        humidity: Relative humidity in % from FastF1 weather data.
        rainfall: True if rain was recorded during this lap.
        total_laps: Total scheduled race laps; used to compute laps_remaining.
        gp_name: GP name matching CIRCUIT_CLUSTER keys (e.g. 'Sakhir').
        mean_sector_speed: Average sector speed in km/h from circuit_features parquet.
            Defaults to prev_speed_st when None.
        prev_deg_rate: Degradation rate from the previous lap (s/lap).
        prev_cum_deg: Cumulative degradation at the previous lap from N09/N10.
        prev_deg_accel: Second derivative of degradation (acceleration, s/lap²).

    Returns:
        PaceOutput with lap_time_pred (s), delta_vs_prev (s), delta_vs_median (s),
        ci_p10 (s), ci_p90 (s), and a reasoning string for N31 LLM synthesis.
    """
    lap_state = build_lap_state(
        driver_number=driver_number, lap_number=lap_number, stint=stint,
        tyre_life=tyre_life, compound=compound, position=position, team=team,
        laps_since_pit=laps_since_pit, fuel_load=fuel_load, year=year,
        prev_lap_time=prev_lap_time, prev_tyre_life=prev_tyre_life,
        prev_speed_st=prev_speed_st, air_temp=air_temp, track_temp=track_temp,
        humidity=humidity, rainfall=rainfall, total_laps=total_laps,
        gp_name=gp_name, mean_sector_speed=mean_sector_speed,
        prev_deg_rate=prev_deg_rate, prev_cum_deg=prev_cum_deg,
        prev_deg_accel=prev_deg_accel,
    )

    lap_time_pred   = predict_lap_time(lap_state)
    delta_vs_prev   = lap_time_pred - prev_lap_time
    p10, p90        = bootstrap_confidence_interval(lap_state)
    median          = session_median_lap_time(gp_name, year, compound, LAPS_REF)
    delta_vs_median = (lap_time_pred - median) if median is not None else float('nan')

    trend   = "faster" if delta_vs_prev < 0 else "slower"
    vs_med  = (f"{delta_vs_median:+.3f}s vs median"
               if median is not None else "no median reference")
    reasoning = (
        f"Lap {lap_number}: predicted {round(lap_time_pred, 3):.3f}s "
        f"({round(delta_vs_prev, 3):+.3f}s, {trend} than prev). "
        f"CI [{round(p10, 1):.1f}–{round(p90, 1):.1f}s]. {vs_med}."
    )

    return PaceOutput(
        lap_time_pred=round(lap_time_pred, 3),
        delta_vs_prev=round(delta_vs_prev, 3),
        delta_vs_median=round(delta_vs_median, 3),
        ci_p10=round(p10, 3),
        ci_p90=round(p90, 3),
        reasoning=reasoning,
    )

In [12]:
# ── Sanity check ─────────────────────────────────────────────────────────────
result = run_pace_agent(
    driver_number=44, lap_number=10, stint=1, tyre_life=10,
    compound='MEDIUM', position=3, team='Mercedes', laps_since_pit=10,
    fuel_load=90, year=2025, prev_lap_time=92.4, prev_tyre_life=9,
    prev_speed_st=310.0, air_temp=28.0, track_temp=42.0,
    humidity=40.0, rainfall=0.0, total_laps=57, gp_name='Barcelona',
)
print(result)

PaceOutput(lap_time_pred=92.803, delta_vs_prev=0.403, delta_vs_median=11.845, ci_p10=90.106, ci_p90=95.278, reasoning='Lap 10: predicted 92.803s (+0.403s, slower than prev). CI [90.1–95.3s]. +11.845s vs median.')


### Step 3 output

`run_pace_agent` returns a `PaceOutput` dataclass with all pace signals in one object:

- `lap_time_pred=92.803s` — absolute predicted lap time for this lap
- `delta_vs_prev=+0.403s` — the model's raw output; slightly slower than the previous lap, consistent with tyre degradation at lap 10
- `delta_vs_median=+11.845s` — large offset because the sanity check uses a synthetic `prev_lap_time=92.4s`, well above a real Barcelona race lap (~81s); will be meaningful in the Step 4 demo with real data
- `ci_p10=90.106s / ci_p90=95.278s` — bootstrap interval; width of ~5s reflects sensitivity to speed trap and temperature noise at 2% perturbation


## Step 4 — Demo: full race replay (lap by lap)

Real-data validation using `laps_featured_2025.parquet`. For a selected GP and driver, we call `run_pace_agent` on every lap of a stint using the actual previous lap time as input (teacher-forcing mode — no error accumulation). This lets us directly compare predicted vs actual lap times.

The demo also shows the confidence interval band across the stint, and reports MAE for the selected driver vs the N06 test-set benchmark (0.392s).

> In production (N31 orchestrator), the agent will run in autoregressive mode: the predicted lap time from lap N becomes `Prev_LapTime` for lap N+1. Teacher-forcing is used here only to cleanly isolate model accuracy from error accumulation.


In [13]:
# ── Step 4: Demo — full race replay (lap by lap) ────────────────────────────
GP_DEMO     = 'Barcelona'
YEAR_DEMO   = 2025
DRIVER_DEMO = 'NOR'

# ── Load laps from parquet ────────────────────────────────────────────────────
laps_all = pd.read_parquet(PROCESSED_DIR / 'laps_featured_2025.parquet')

# ── Get driver number via FastF1, cast to int to match parquet ───────────────
session = fastf1.get_session(YEAR_DEMO, GP_DEMO, 'R')
session.load(laps=True, telemetry=False, weather=False, messages=False)

abbr_map   = (session.laps[['DriverNumber', 'Driver']]
              .drop_duplicates()
              .set_index('Driver')['DriverNumber']
              .to_dict())
driver_num = int(abbr_map[DRIVER_DEMO])

# ── Filter: selected driver + GP, no rainfall ────────────────────────────────
mask = (
    (laps_all['GP_Name']                  == GP_DEMO) &
    (laps_all['DriverNumber'].astype(int) == driver_num) &
    (laps_all['Rainfall']                 == 0.0)
)
driver_laps = laps_all[mask].sort_values('LapNumber').reset_index(drop=True)

print(f"Driver {DRIVER_DEMO} (#{driver_num}) — {GP_DEMO} {YEAR_DEMO}")
print(f"Laps in parquet: {len(driver_laps)}")

if len(driver_laps) == 0:
    print("WARNING: no laps found — check GP_Name or driver number")
else:
    total_laps = int(driver_laps['LapNumber'].max() + driver_laps['laps_remaining'].iloc[-1])
    print(f"Total race laps (estimated): {total_laps}")
    print(driver_laps[['LapNumber', 'Stint', 'Compound', 'TyreLife', 'LapTime_s']].head(10).to_string())

req         WARNING 	DEFAULT CACHE ENABLED! (7.0 MB) C:\Users\victo\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Spanish Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 19 drivers: ['81', '4', '16', '63', '27', '44', '6', '10', '14', '1', '30', '5', '22', '55', '43', '31', '87', '12', '23']


Driver NOR (#4) — Barcelona 2025
Laps in parquet: 55
Total race laps (estimated): 66
   LapNumber  Stint Compound  TyreLife  LapTime_s
0        2.0    1.0     SOFT       2.0     80.904
1        3.0    1.0     SOFT       3.0     80.690
2        4.0    1.0     SOFT       4.0     80.800
3        5.0    1.0     SOFT       5.0     80.644
4        6.0    1.0     SOFT       6.0     80.713
5        7.0    1.0     SOFT       7.0     80.837
6        8.0    1.0     SOFT       8.0     80.902
7        9.0    1.0     SOFT       9.0     80.950
8       10.0    1.0     SOFT      10.0     80.929
9       11.0    1.0     SOFT      11.0     80.959


In [ ]:
# ── Step 4: Race replay — live updating plot ─────────────────────────────────
REPLAY_DELAY = 0   # seconds between laps; set to 0 to skip delay
N_AHEAD      = 12  # future laps to preview at each replay step

COMPOUND_COLORS = {
    'SOFT':         '#E8002D',
    'MEDIUM':       '#FFF200',
    'HARD':         '#EEEEEE',
    'INTERMEDIATE': '#39B54A',
    'WET':          '#0067FF',
}

In [ ]:
def _make_prediction(row):
    """Run run_pace_agent for one driver_laps row and return a result dict."""
    output = run_pace_agent(
        driver_number     = int(row['DriverNumber']),
        lap_number        = int(row['LapNumber']),
        stint             = int(row['Stint']),
        tyre_life         = int(row['TyreLife']),
        compound          = str(row['Compound']),
        position          = int(row['Position']) if not pd.isna(row.get('Position', np.nan)) else 10,
        team              = str(row['Team']),
        laps_since_pit    = int(row.get('LapsSincePitStop', row['TyreLife'])),
        fuel_load         = float(row.get('FuelLoad', 80.0)),
        year              = int(row['Year']),
        prev_lap_time     = float(row['Prev_LapTime']),
        prev_tyre_life    = int(row.get('Prev_TyreLife', row['TyreLife'] - 1)),
        prev_speed_st     = float(row.get('Prev_SpeedST', 305.0)),
        air_temp          = float(row.get('AirTemp', 25.0)),
        track_temp        = float(row.get('TrackTemp', 40.0)),
        humidity          = float(row.get('Humidity', 50.0)),
        rainfall          = float(row.get('Rainfall', 0.0)),
        total_laps        = total_laps,
        gp_name           = str(row['GP_Name']),
        mean_sector_speed = float(row['mean_sector_speed']) if not pd.isna(row.get('mean_sector_speed', np.nan)) else None,
        prev_deg_rate     = float(row.get('Prev_DegradationRate', 0.0)),
        prev_cum_deg      = float(row.get('Prev_CumulativeDeg', 0.0)),
        prev_deg_accel    = float(row.get('Prev_DegAcceleration', 0.0)),
    )
    return {
        'LapNumber': int(row['LapNumber']),
        'Actual':    float(row['LapTime_s']),
        'Pred':      output.lap_time_pred,
        'CI_P10':    output.ci_p10,
        'CI_P90':    output.ci_p90,
        'Compound':  str(row['Compound']),
        'Error':     output.lap_time_pred - float(row['LapTime_s']),
    }


def _compute_stint_bounds(valid_rows):
    """Return lap numbers where a stint change occurs, for vertical dividers."""
    bounds, prev_s = [], None
    for _, r in valid_rows.iterrows():
        s = int(r['Stint'])
        if prev_s is not None and s != prev_s:
            bounds.append(int(r['LapNumber']))
        prev_s = s
    return bounds


def run_replay_demo(valid_rows):
    """Pre-compute all predictions and run the live-updating lap-by-lap replay plot.

    Iterates valid_rows twice: once to build the full prediction DataFrame, then
    once per lap to render the cumulative actual-vs-predicted plot with a rolling
    CI band and a future-lookahead window. Prints a summary table at the end.

    Args:
        valid_rows: Filtered pd.DataFrame of driver laps with non-null
            Prev_LapTime (≥ 60 s) and LapTime_s. Produced by filtering
            driver_laps before calling this function.

    Returns:
        pd.DataFrame of all predictions with columns LapNumber, Actual, Pred,
        CI_P10, CI_P90, Compound, Error.
    """
    print(f"Pre-computing {len(valid_rows)} predictions...")
    all_preds_df = pd.DataFrame([_make_prediction(r) for _, r in valid_rows.iterrows()])
    print("Done. Starting replay...")

    stint_bounds = _compute_stint_bounds(valid_rows)
    shown_laps   = []
    fig          = None

    for cur_lap in all_preds_df['LapNumber']:
        shown_laps.append(cur_lap)

        df_past   = all_preds_df[all_preds_df['LapNumber'].isin(shown_laps)]
        df_future = all_preds_df[all_preds_df['LapNumber'] > cur_lap].head(N_AHEAD)

        clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
        fig.suptitle(
            f'{DRIVER_DEMO} — {GP_DEMO} {YEAR_DEMO}  │  Lap {cur_lap} / {total_laps}',
            fontsize=13, fontweight='bold', y=0.98,
        )

        # ── Subplot 1: actual vs predicted ────────────────────────────────────
        ax1.fill_between(
            df_past['LapNumber'], df_past['CI_P10'], df_past['CI_P90'],
            alpha=0.18, color='steelblue', label='CI P10–P90 (past)',
        )
        ax1.plot(
            df_past['LapNumber'], df_past['Actual'],
            '--', color='#555555', linewidth=1.6, alpha=0.85, label='Actual',
        )
        ax1.plot(
            df_past['LapNumber'], df_past['Pred'],
            '-', color='steelblue', linewidth=1.4, alpha=0.7, zorder=2,
        )
        for compound, grp in df_past.groupby('Compound'):
            color = COMPOUND_COLORS.get(compound, 'black')
            ax1.scatter(
                grp['LapNumber'], grp['Pred'],
                color=color, s=45, zorder=3,
                edgecolors='black', linewidths=0.4,
                label=f'Pred ({compound})',
            )
        if not df_future.empty:
            ax1.plot(
                df_future['LapNumber'], df_future['Pred'],
                '--', color='steelblue', linewidth=1.4, alpha=0.30,
                label=f'Future pred (next {N_AHEAD})',
            )
            ax1.fill_between(
                df_future['LapNumber'], df_future['CI_P10'], df_future['CI_P90'],
                alpha=0.07, color='steelblue',
            )
        for sb in stint_bounds:
            ax1.axvline(sb - 0.5, color='black', linestyle=':', linewidth=0.9, alpha=0.5)
        ax1.set_ylabel('Lap time (s)', fontsize=10)
        ax1.legend(loc='upper right', fontsize=8, ncol=4, framealpha=0.85)
        ax1.grid(True, alpha=0.22)
        ax1.tick_params(axis='x', which='both', bottom=False)

        # ── Subplot 2: per-lap error ───────────────────────────────────────────
        ax2.axhline(0, color='black', linewidth=0.9)
        ax2.axhspan(-0.5, 0.5, alpha=0.07, color='green', label='±0.5s band')
        ax2.plot(
            df_past['LapNumber'], df_past['Error'],
            '-', color='steelblue', linewidth=0.8, alpha=0.4, zorder=2,
        )
        for compound, grp in df_past.groupby('Compound'):
            color = COMPOUND_COLORS.get(compound, 'black')
            ax2.scatter(
                grp['LapNumber'], grp['Error'],
                color=color, s=45, zorder=3,
                edgecolors='black', linewidths=0.4,
            )
        for sb in stint_bounds:
            ax2.axvline(sb - 0.5, color='black', linestyle=':', linewidth=0.9, alpha=0.5)
        current_mae = df_past['Error'].abs().mean()
        ax2.text(
            0.01, 0.95, f'MAE so far: {current_mae:.3f}s',
            transform=ax2.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8),
        )
        ax2.set_xlabel('Lap number', fontsize=10)
        ax2.set_ylabel('Error  (pred − actual, s)', fontsize=10)
        ax2.legend(fontsize=8, framealpha=0.85)
        ax2.grid(True, alpha=0.22)

        plt.tight_layout()
        plt.show()

        if REPLAY_DELAY > 0:
            time.sleep(REPLAY_DELAY)

    # ── Final summary ──────────────────────────────────────────────────────────
    df_results   = all_preds_df.copy()
    mae          = df_results['Error'].abs().mean()
    bias         = df_results['Error'].mean()
    ci_coverage  = (
        (df_results['Actual'] >= df_results['CI_P10']) &
        (df_results['Actual'] <= df_results['CI_P90'])
    ).mean() * 100

    print(f"\n{'─'*55}")
    print(f"N25 Pace Agent — {GP_DEMO} {YEAR_DEMO} ({DRIVER_DEMO})")
    print(f"{'─'*55}")
    print(f"  Laps evaluated : {len(df_results)}")
    print(f"  MAE            : {mae:.3f}s   (N06 test benchmark: 0.392s)")
    print(f"  Bias           : {bias:+.3f}s")
    print(f"  CI coverage    : {ci_coverage:.1f}%  (target: ~80%)")
    print(f"{'─'*55}")

    if fig is not None:
        plot_path = OUTPUTS_DIR / f'N25_replay_{DRIVER_DEMO}_{GP_DEMO}_{YEAR_DEMO}.png'
        fig.savefig(plot_path, dpi=120, bbox_inches='tight')
        print(f"  Plot saved  → {plot_path.relative_to(repo_root)}")

    return df_results

In [ ]:
valid_rows = driver_laps[
    driver_laps['Prev_LapTime'].notna() &
    (driver_laps['Prev_LapTime'] >= 60) &
    driver_laps['LapTime_s'].notna()
]

df_results = run_replay_demo(valid_rows)

---

## Step 5 — LangGraph ReAct agent

This step wraps the inference functions from Steps 1–3 into a proper LangGraph ReAct agent — the interface that N31 (Strategy Orchestrator) will use to delegate pace queries.

**Architecture**:
- `@tool`-decorated functions expose `predict_pace_tool` and `get_session_median_tool` to the LLM
- `create_react_agent` (LangGraph prebuilt) gives the LLM a reasoning loop: it can call tools in sequence, inspect their outputs, and produce a final answer
- `ChatOpenAI` works for both OpenAI API and LM Studio — only `base_url` and `api_key` differ

The key difference from `run_pace_agent` in Step 3: the ReAct agent lets the LLM **decide** which tools to call and can reason about the outputs before responding. For a deterministic single-tool workflow the two are equivalent; the agent pattern pays off in N31 where the supervisor LLM selects which sub-agents to activate each lap.

> **Dependencies**: `pip install langchain langchain-openai langgraph`  
> **Local LLM**: set `LLM_PROVIDER = 'lmstudio'` (LM Studio must be running on `localhost:1234`)

In [15]:
# ── Step 5: LangGraph ReAct agent ───────────────────────────────────────────



# ── @tool-decorated inference functions ─────────────────────────────────────
@lc_tool
def predict_pace_tool(
    driver_number: int, lap_number: int, stint: int, tyre_life: int,
    compound: str, position: int, team: str, laps_since_pit: int,
    fuel_load: float, year: int, prev_lap_time: float, prev_tyre_life: int,
    prev_speed_st: float, air_temp: float, track_temp: float,
    humidity: float, rainfall: float, total_laps: int, gp_name: str,
) -> dict:
    """Predict the absolute lap time (seconds) for the current lap using the N06 XGBoost model.

    Call this whenever a lap time prediction is needed for pace or strategy analysis.
    Delegates to run_pace_agent with default values for prev_deg_rate/cum_deg/accel.

    Args:
        driver_number: Car number used to look up TeamID encoding.
        lap_number: Current race lap number (1-indexed).
        stint: Stint number (1-indexed).
        tyre_life: Laps on the current tyre set.
        compound: Pirelli compound string ('SOFT', 'MEDIUM', 'HARD', etc.).
        position: Current race position (1-based).
        team: Team name matching TEAM_ID encoding map (e.g. 'McLaren').
        laps_since_pit: Laps elapsed since the last pit stop.
        fuel_load: Fuel fraction in [0, 1] (laps_remaining / total_laps).
        year: Race year integer (2023/2024/2025).
        prev_lap_time: Previous lap time in seconds.
        prev_tyre_life: TyreLife on the previous lap.
        prev_speed_st: Speed trap reading in km/h from the previous lap.
        air_temp: Air temperature in °C.
        track_temp: Track surface temperature in °C.
        humidity: Relative humidity in %.
        rainfall: True if rain was recorded during this lap.
        total_laps: Total scheduled race laps.
        gp_name: GP name matching CIRCUIT_CLUSTER keys (e.g. 'Sakhir').

    Returns:
        Dict with keys:
        - lap_time_pred (float, seconds): absolute predicted lap time.
        - delta_vs_prev (float, seconds): difference vs previous lap; negative = faster.
        - delta_vs_median (float, seconds): difference vs historical GP/compound median; NaN if unavailable.
        - ci_p10 (float, seconds): P10 bootstrap confidence bound on lap_time_pred.
        - ci_p90 (float, seconds): P90 bootstrap confidence bound on lap_time_pred.
    """
    out = run_pace_agent(
        driver_number=driver_number, lap_number=lap_number, stint=stint,
        tyre_life=tyre_life, compound=compound, position=position, team=team,
        laps_since_pit=laps_since_pit, fuel_load=fuel_load, year=year,
        prev_lap_time=prev_lap_time, prev_tyre_life=prev_tyre_life,
        prev_speed_st=prev_speed_st, air_temp=air_temp, track_temp=track_temp,
        humidity=humidity, rainfall=rainfall, total_laps=total_laps, gp_name=gp_name,
    )
    return {
        'lap_time_pred':   out.lap_time_pred,
        'delta_vs_prev':   out.delta_vs_prev,
        'delta_vs_median': out.delta_vs_median,
        'ci_p10':          out.ci_p10,
        'ci_p90':          out.ci_p90,
    }


@lc_tool
def get_session_median_tool(gp_name: str, year: int, compound: str) -> dict:
    """Return the historical median lap time (seconds) for a GP / year / compound combination.

    Use this as a reference baseline to contextualise a predicted lap time from
    predict_pace_tool. The median is computed from IsAccurate laps in the N06
    training parquet filtered to non-SC, non-VSC conditions.

    Args:
        gp_name: GP name matching the parquet GP_Name column (e.g. 'Sakhir', 'Silverstone').
        year: Race year integer (2023/2024/2025).
        compound: Pirelli compound string ('SOFT', 'MEDIUM', 'HARD').

    Returns:
        Dict with key:
        - median_lap_time (float, seconds): historical median lap time for this combination.
          Value is NaN (float('nan')) when no matching laps are found.
    """
    median = session_median_lap_time(gp_name, year, compound, LAPS_REF)
    return {'median_lap_time': median if median is not None else float('nan')}

In [16]:
# ── LLM config ───────────────────────────────────────────────────────────────
LLM_PROVIDER = 'lmstudio'   # 'openai' | 'lmstudio'

if LLM_PROVIDER == 'lmstudio':
    LLM = ChatOpenAI(
        model='local-model',
        base_url='http://localhost:1234/v1',
        api_key='lmstudio',
        temperature=0,
    )
else:
    LLM = ChatOpenAI(model='gpt-4o-mini', temperature=0)

In [17]:
# ── System prompt ─────────────────────────────────────────────────────────────
PACE_SYSTEM_PROMPT = """You are the Pace Agent in an F1 race strategy system.

Your responsibility: answer the question "how fast is this car going this lap?"

Tools available:
- `predict_pace_tool` — predicts absolute lap time using the N06 XGBoost model
- `get_session_median_tool` — returns the historical median for this GP/compound as a baseline

Always call `predict_pace_tool` first, then `get_session_median_tool` to contextualise the result.
Respond with a concise JSON summary: lap_time_pred, delta_vs_prev, delta_vs_median, ci_p10, ci_p90.
Never invent numbers — use only the values returned by the tools."""


# ── Create ReAct agent ────────────────────────────────────────────────────────
PACE_TOOLS = [predict_pace_tool, get_session_median_tool]
pace_react_agent = create_react_agent(
    model=LLM,
    tools=PACE_TOOLS,
    prompt=PACE_SYSTEM_PROMPT,
)

print("Pace ReAct agent created.")
print(f"LLM provider : {LLM_PROVIDER}  |  model: {LLM.model_name}")
print(f"Tools        : {[t.name for t in PACE_TOOLS]}")

Pace ReAct agent created.
LLM provider : lmstudio  |  model: local-model
Tools        : ['predict_pace_tool', 'get_session_median_tool']


C:\Users\victo\AppData\Local\Temp\ipykernel_15684\2954358627.py:17: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  pace_react_agent = create_react_agent(


In [18]:
# ── Demo: invoke the ReAct agent via natural language ────────────────────────
demo_query = """Predict the pace for the following race state:
- Driver 4 (LAN), McLaren, Barcelona 2025
- Lap 32, Stint 2, TyreLife 15, Compound MEDIUM, Position 1
- PrevLapTime 82.1s, PrevTyreLife 14, PrevSpeedST 315.0 km/h
- FuelLoad 55 kg, Air 26°C, Track 38°C, Humidity 45%, Rainfall 0
- TotalLaps 66
"""

try:
    response = pace_react_agent.invoke({"messages": [("user", demo_query)]})
    print("─── Reasoning trace ──────────────────────────────────────────────")
    for msg in response['messages']:
        name    = msg.__class__.__name__
        content = str(msg.content)[:500] if msg.content else ''
        if content:
            print(f"[{name}]\n{content}\n")
    print("──────────────────────────────────────────────────────────────────")

except Exception as e:
    print("[LLM unavailable — configure LLM_PROVIDER and API key to run the ReAct demo]")
    print(f"Error: {e}")
    print("\nFalling back to direct run_pace_agent call:")
    out = run_pace_agent(
        driver_number=4, lap_number=32, stint=2, tyre_life=15,
        compound='MEDIUM', position=1, team='McLaren', laps_since_pit=15,
        fuel_load=55, year=2025, prev_lap_time=82.1, prev_tyre_life=14,
        prev_speed_st=315.0, air_temp=26.0, track_temp=38.0,
        humidity=45.0, rainfall=0.0, total_laps=66, gp_name='Barcelona',
    )
    print(out)

─── Reasoning trace ──────────────────────────────────────────────
[HumanMessage]
Predict the pace for the following race state:
- Driver 4 (LAN), McLaren, Barcelona 2025
- Lap 32, Stint 2, TyreLife 15, Compound MEDIUM, Position 1
- PrevLapTime 82.1s, PrevTyreLife 14, PrevSpeedST 315.0 km/h
- FuelLoad 55 kg, Air 26°C, Track 38°C, Humidity 45%, Rainfall 0
- TotalLaps 66


[AIMessage]
```json
{
  "lap_time_pred": 79.5,
  "delta_vs_prev": -2.6,
  "delta_vs_median": NaN,
  "ci_p10": 78.8,
  "ci_p90": 80.2
}
```

──────────────────────────────────────────────────────────────────


---

## Step 6 — Export schema & config

Exports a JSON config for the Pace Agent to `data/models/agents/`. This file is the contract between N25 and N31 (Strategy Orchestrator): it describes model paths, encoding artifact locations, inference hyperparameters, the output schema, and the LangGraph tool names.

N31 reads these configs at startup to discover and load each sub-agent without importing the notebooks directly.

In [19]:
# ── Step 6: Export agent config ─────────────────────────────────────────────
AGENTS_DIR = repo_root / 'data' / 'models' / 'agents'
AGENTS_DIR.mkdir(parents=True, exist_ok=True)

pace_agent_config = {
    'agent':   'N25_pace_agent',
    'version': 'v1',
    'model': {
        'type':       'xgboost',
        'path':       str(MODELS_DIR / 'xgb_laptime_delta_final.json'),
        'features':   str(MODELS_DIR / 'xgb_laptime_delta_feature_names.json'),
        'n_features': len(FEATURES),
    },
    'artifacts': {
        'compound_encoding': str(FEATURE_MANIFEST),
        'circuit_clusters':  str(CLUSTER_PARQUET),
        'team_encoding':     str(LAPS_FEATURED),
        'reference_laps':    str(LAPS_FEATURED),
    },
    'inference': {
        'bootstrap_n': N_BOOTSTRAP,
        'noise_pct':   0.02,
        'ci_bounds':   [10, 90],
    },
    'output_schema': {
        'lap_time_pred':   'float — absolute predicted lap time (s)',
        'delta_vs_prev':   'float — delta vs Prev_LapTime (s)',
        'delta_vs_median': 'float — delta vs GP/compound session median (s)',
        'ci_p10':          'float — bootstrap P10 bound (s)',
        'ci_p90':          'float — bootstrap P90 bound (s)',
    },
    'langgraph': {
        'tools':       ['predict_pace_tool', 'get_session_median_tool'],
        'llm':         'ChatOpenAI — openai / lmstudio',
        'entry_point': 'run_pace_agent',
    },
}

config_path = AGENTS_DIR / 'pace_agent_config_v1.json'
config_path.write_text(json.dumps(pace_agent_config, indent=2))
print(f"Config exported → {config_path.relative_to(repo_root)}")
print(json.dumps(pace_agent_config, indent=2))

Config exported → data\models\agents\pace_agent_config_v1.json
{
  "agent": "N25_pace_agent",
  "version": "v1",
  "model": {
    "type": "xgboost",
    "path": "c:\\Users\\victo\\Desktop\\Documents\\Cuarto A\u00f1o\\TFG\\F1_Strat_Manager\\data\\models\\lap_time\\xgb_laptime_delta_final.json",
    "features": "c:\\Users\\victo\\Desktop\\Documents\\Cuarto A\u00f1o\\TFG\\F1_Strat_Manager\\data\\models\\lap_time\\xgb_laptime_delta_feature_names.json",
    "n_features": 25
  },
  "artifacts": {
    "compound_encoding": "c:\\Users\\victo\\Desktop\\Documents\\Cuarto A\u00f1o\\TFG\\F1_Strat_Manager\\data\\processed\\feature_manifest_laptime.json",
    "circuit_clusters": "c:\\Users\\victo\\Desktop\\Documents\\Cuarto A\u00f1o\\TFG\\F1_Strat_Manager\\data\\processed\\circuit_clustering\\circuit_clusters_k4_2025.parquet",
    "team_encoding": "c:\\Users\\victo\\Desktop\\Documents\\Cuarto A\u00f1o\\TFG\\F1_Strat_Manager\\data\\processed\\laps_featured_2025.parquet",
    "reference_laps": "c:\\Use